# Redis Manager Usage Examples

This notebook demonstrates how to use the `RedisDB` manager class to create, manage, and interact with a Redis database in a Docker container.

## Setup

### Import Dependencies

In [ ]:
import sys
import uuid
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from docker_db.dbs.redis_db import RedisConfig, RedisDB

## Create Redis Configuration

In [ ]:
temp_dir = Path(tempfile.mkdtemp())

container_name = f"demo-redis-{uuid.uuid4().hex[:8]}"

config = RedisConfig(
    database=0,
    project_name="demo",
    container_name=container_name,
    workdir=temp_dir,
    retries=20,
    delay=2,
)

db_manager = RedisDB(config)

## Start Redis

In [3]:
db_manager.create_db()
print(f"Redis started in container: {container_name}")
print(f"Connection string: {db_manager.connection_string()}")

Redis started in container: demo-redis-d7f1c3d8
Connection string: redis://127.0.0.1:6379/0


## Write and Read Data

In [4]:
client = db_manager.connection

client.set("app:status", "running")
client.hset("user:1", mapping={"name": "alice", "role": "admin"})
client.rpush("events", "created", "updated", "completed")

print("app:status ->", client.get("app:status"))
print("user:1 ->", client.hgetall("user:1"))
print("events ->", client.lrange("events", 0, -1))

app:status -> running
user:1 -> {'name': 'alice', 'role': 'admin'}
events -> ['created', 'updated', 'completed']


## Clean Up

In [5]:
db_manager.delete_db(running_ok=True)
print(f"Redis container '{container_name}' deleted")

Container demo-redis-d7f1c3d8 stopped and port 6379 is free
Redis container 'demo-redis-d7f1c3d8' deleted


## Conclusion

This notebook demonstrated how to:

1. Configure and create a Redis instance with `RedisDB`
2. Connect using redis-py
3. Write and read key-value, hash, and list data
4. Clean up the container when finished